In [16]:
import os
from llama_parse import LlamaParse
from langchain_core.documents import Document as LC_Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import weaviate
from weaviate.classes.config import Configure, Property, DataType


Load the Data and Parsing the data

In [ ]:
api_lp_key = "llx-TwgXg8jk0bjNKagkxevsUqBqUdYWeaj5zILMuX0kL04xhm9w"

parser = LlamaParse(
    api_key = api_lp_key,
    result_type = "markdown", # For simple documents you can text instead of markdown. Markdown is more reliable when reading the tables and images
    num_workers = 2,
    verbose = True,
    language = "en"
)

file_path = "./file_survey_paper.pdf"
documents = parser.load_data(file_path)


Started parsing the file under job_id 05c885ff-ade4-4660-9f79-103ae7c8c836


In [23]:
print(f"Loaded {len(documents)} document pages/sections.")
print(f"Sample Content:\n{documents[0].text[:500]}...")

Loaded 144 document pages/sections.
Sample Content:
arXiv:2303.18223v19 [cs.CL] 18 Mar 2026

# A Survey of Large Language Models

Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Yang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang, Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†

# Abstract

Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence by machine. L...


Using recursive splitting and Semantic chunking

In [24]:
embed_model = OllamaEmbeddings(model="nomic-embed-text")


# # convert LlamaIndex Documents -> LangChain Documents
# lc_documents = [
#     LC_Document(
#         page_content=doc.text,
#         metadata=doc.metadata
#     )
#     for doc in documents
# ]

# Convert LlamaIndex Documents → LangChain Documents with explicit metadata mapping
lc_documents = []
for doc in documents:
    # LlamaParse usually uses 'page_label' for the human-readable page number
    page_num = doc.metadata.get("page_label", "Unknown")
    
    lc_documents.append(
        LC_Document(
            page_content=doc.text,
            metadata={
                "source": doc.metadata.get("file_name", "Unknown"),
                "page_number": page_num  # Explicitly store it here
            }
        )
    )

# Before doing the semantic chunking. I am doing additional chunking. so, a chunk can't have more than 8192 tokens. which is maximum for model.
pre_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,       
    chunk_overlap=400,     # small overlap to preserve context at boundaries
)
pre_split_docs = pre_splitter.split_documents(lc_documents)
print(f"Pre-split into {len(pre_split_docs)} chunks before semantic chunking.")


chunks = SemanticChunker(
    embed_model,
    breakpoint_threshold_type="percentile", 
    breakpoint_threshold_amount=95.0         # tune: lower = more chunks, higher = fewer
)


semantic_chunks = chunks.split_documents(pre_split_docs)

Pre-split into 288 chunks before semantic chunking.


In [25]:
print(f"Created {len(semantic_chunks)} semantic chunks using nomic-embed-text.")
print(f"Example Chunk Content:\n{semantic_chunks[0].page_content[:200]}...")

Created 1063 semantic chunks using nomic-embed-text.
Example Chunk Content:
arXiv:2303.18223v19 [cs.CL] 18 Mar 2026

# A Survey of Large Language Models

Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zi...


Generating Embeddings and Using the Matryoshka Representation Learning(MRL) 

In [26]:
#We are using MRL to reduce the dimension of embeddings (To save space in the database and easy for search and to retreive the chunks from database)

Target_num = 256

processed_chunks = []
print(f"Slicing vectors to {Target_num} dimensions using MRL")

for chunk in semantic_chunks:
    full_vector = embed_model.embed_query(chunk.page_content)
    #we pulling the model from the Ollama to generate the embeddings
    sliced_vector = np.array(full_vector[:Target_num])
    #MRL slicing. reducing from standard dimension to 256 dimension

    #we have to do renormalization for sliced vectors
    norm = np.linalg.norm(sliced_vector)
    if norm > 0:
        normalized_vector = sliced_vector / norm
    else:
        normalized_vector = sliced_vector

    #store the result with its metadata
    processed_chunks.append({
        "content": chunk.page_content,
        "metadata": chunk.metadata,
        "vector": normalized_vector.tolist()
    })


Slicing vectors to 256 dimensions using MRL


In [29]:
print(f"Ready for Weaviate: {len(processed_chunks)} vectors at {Target_num} dims.")

Ready for Weaviate: 1063 vectors at 256 dims.
